MLflow Tracking

In [1]:
import mlflow
import mlflow.sklearn
import pandas as pd
import joblib
import os

# Set paths
BASE = os.path.dirname(os.path.abspath("__file__"))

mlflow.set_experiment("RetailPulse-ChurnModel")

with mlflow.start_run(run_name="xgb_churn_v1"):
    # Load model
    model = joblib.load("../models/xgb_churn_model.joblib")
    
    # Load data
    df = pd.read_csv("../data/processed/customer_features.csv")
    
    # Log parameters
    mlflow.log_param("model_type", "XGBoost")
    mlflow.log_param("n_features", len(df.columns))
    mlflow.log_param("n_customers", len(df))
    
    # Log metrics
    mlflow.log_metric("auc_roc", 0.91)
    mlflow.log_metric("churn_rate", 
        round((df['CustomerActivityStatus'] == 'Churned').mean(), 3))
    
    print("✅ MLflow run logged successfully!")
    print(f"Total customers: {len(df)}")

e:\Priyanshu\OneDrive\Documents\Zidio Internship\Project file\RetailPulse\RetailPulse-main\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026/05/30 12:12:09 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/05/30 12:12:09 INFO mlflow.store.db.utils: Updating database tables
2026/05/30 12:12:11 INFO mlflow.tracking.fluent: Experiment with name 'RetailPulse-ChurnModel' does not exist. Creating a new experiment.


✅ MLflow run logged successfully!
Total customers: 5878


In [8]:
import os
os.makedirs("../reports", exist_ok=True)
print("✅ reports folder created")

✅ reports folder created


In [ ]:
Evidently drift detection

In [9]:
import pandas as pd
import numpy as np
from evidently import Report
from evidently.presets import DataDriftPreset

# Load data
df = pd.read_csv("../data/processed/customer_features.csv")

# Select numeric columns only
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
df_numeric = df[numeric_cols]

# Split into reference (first half) and current (second half)
reference = df_numeric.iloc[:len(df_numeric)//2].reset_index(drop=True)
current = df_numeric.iloc[len(df_numeric)//2:].reset_index(drop=True)

print(f"Reference data: {len(reference)} customers")
print(f"Current data: {len(current)} customers")

# Generate drift report
report = Report([DataDriftPreset()])
snapshot = report.run(reference_data=reference, current_data=current)

# Save report
snapshot.save_html("../reports/data_drift_report.html")
print("✅ Drift report saved to reports/data_drift_report.html")

Reference data: 2939 customers
Current data: 2939 customers
✅ Drift report saved to reports/data_drift_report.html


MLflow forecast tracking

In [10]:
mlflow.set_experiment("RetailPulse-Forecasting")

with mlflow.start_run(run_name="prophet_forecast_v1"):
    forecast = pd.read_csv("../data/processed/prophet_forecast_30d.csv")
    
    mlflow.log_param("model_type", "Prophet")
    mlflow.log_param("forecast_horizon_days", 30)
    mlflow.log_metric("avg_predicted_revenue", 
        round(forecast['yhat'].mean(), 2))
    mlflow.log_metric("max_predicted_revenue", 
        round(forecast['yhat'].max(), 2))
    
    print("✅ Forecast run logged to MLflow!")
    print(f"Average predicted daily revenue: £{forecast['yhat'].mean():.2f}")

2026/05/30 12:16:27 INFO mlflow.tracking.fluent: Experiment with name 'RetailPulse-Forecasting' does not exist. Creating a new experiment.


✅ Forecast run logged to MLflow!
Average predicted daily revenue: £13915.25
